# Mission 3: The Red Bag

The case file says Aarav owns a red backpack. Find red-colored candidate regions in the staged scene. A color mask finds color, never a person's identity or ownership.

## Objectives and setup
Compare BGR/RGB/HSV, make a two-range red mask, clean it morphologically, extract contours and inspect false positives. From repository root install with `python -m pip install -e ".[test]"`, then open in Jupyter Lab. Inputs include the staged bag, low-light and warm-background scenes, multiple red distractors, partial occlusion and real photos. Headless OpenCV has no GUI trackbars; adjust the numeric parameters in the function calls.

Exercise: predict why one red hue interval is not enough before viewing both masks.

In [ ]:
from pathlib import Path
import sys, cv2
import matplotlib.pyplot as plt
ROOT = Path.cwd()
if not (ROOT / 'src').exists(): ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / 'src'))
from the_last_chai.image_enhancement import load_image, save_image
from the_last_chai.color_detection import red_mask, clean_mask, find_red_candidates, draw_candidates
image = load_image(str(ROOT / 'assets/images/red_bag_scene.png'))
mask = red_mask(image)
plt.figure(figsize=(12,4)); plt.subplot(1,2,1); plt.imshow(cv2.cvtColor(image, cv2.COLOR_BGR2RGB)); plt.axis('off'); plt.subplot(1,2,2); plt.imshow(mask, cmap='gray'); plt.title('Red pixels'); plt.axis('off');

OpenCV's 8-bit HSV hue spans 0-179. Red wraps around the ends, so the reusable module joins hue intervals 0-10 and 170-179. Saturation/value thresholds reduce dull shadows. Opening removes small specks; closing fills small gaps. Contours outline connected foreground regions. A candidate box is not an object identity.

Exercise: lower `min_area` and note which extra red regions appear. Shadows and labels can be false positives; occlusion can split a real bag.

In [ ]:
mask, candidates = find_red_candidates(image, min_area=500, kernel_size=5)
annotated = draw_candidates(image, candidates)
print('Candidate boxes (largest first):', candidates)
plt.figure(figsize=(12,4)); plt.imshow(cv2.cvtColor(annotated, cv2.COLOR_BGR2RGB)); plt.axis('off');
save_image(str(ROOT / 'outputs/level3_candidates.png'), annotated)

In [ ]:
# TODO: Lower min_area or raise low_value. Which false positives change?
trial_mask, trial_candidates = find_red_candidates(
    load_image(str(ROOT / 'assets/images/red_objects_distractors.png')),
    min_area=300,
    low_value=40,
)
print('Your candidate count:', len(trial_candidates))
plt.figure(figsize=(8, 4)); plt.imshow(trial_mask, cmap='gray'); plt.title('Your adjusted HSV mask'); plt.axis('off');

Try the similar-color scene and partially obscured frame. Trackbars are not available in headless OpenCV; change `low_saturation`, `low_value`, `min_area` in the function call instead. Hint 1: inspect the grayscale mask. Hint 2: adjust hue intervals in `color_detection.py`. Hint 3: compare area and box location to the ground-truth scene, not a generic red rectangle.

Clue: **BACKPACK OWNER: AARAV** comes from the case file. Challenge: list one false positive the detector may produce. Common errors: using RGB conversion on a BGR image; treating a mask as a multi-channel color image; setting area too high.

In [ ]:
similar = load_image(str(ROOT / 'assets/images/similar_colour_scene.png'))
partial = load_image(str(ROOT / 'assets/images/partially_obscured_bag.png'))
for label, frame in [('similar colors', similar), ('occlusion', partial)]:
    _, found = find_red_candidates(frame, min_area=500)
    print(label, 'candidate count:', len(found), 'largest:', found[0] if found else 'none')

## Recap
HSV segmentation, morphology and contours locate red candidates. A human compares their position with staged ground truth; ownership is an independent case-file clue. The same module is used in mission four.